# Capítulo 7 — Modelos Predictivos de Trayectoria Académica

Este cuaderno visualiza los resultados del pipeline de Machine Learning (`29_modelos_predictivos.py`).
Se comparan dos enfoques de clasificación multiclase con **Random Forest**:

| Modelo | Predictores | Objetivo |
|--------|-------------|----------|
| **A — Alerta Temprana** | S1 a S4 | Detectar riesgo a mitad de carrera |
| **B — Trayectoria Completa** | S1 a S8 | Predicción con trayectoria completa |

Para cada modelo se reportan:
- Matriz de Confusión (validación cruzada 5-fold estratificada)
- Importancia de variables (*Feature Importance*)
- **IC al 95 %** por bootstrapping (B = 2000)
- **Prueba de hipótesis McNemar** entre modelos

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')

BASE_DIR  = Path('../../').resolve()
IMG_DIR   = BASE_DIR / 'imagenes'
JSON_PATH = Path('metricas_modelos_predictivos.json')

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.2)

# Cargar métricas
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    metrics = json.load(f)

m_A = metrics['modelo_A']
m_B = metrics['modelo_B']
mc  = metrics['mcnemar']

print('✅ Métricas cargadas correctamente')
print(f"  Modelo A: {m_A['n_obs']} observaciones (S1-S{m_A['n_semestres']})")
print(f"  Modelo B: {m_B['n_obs']} observaciones (S1-S{m_B['n_semestres']})")

---
## 1. Desempeño General — Accuracy y F1-Score con IC al 95 %

Los intervalos de confianza se estimaron por **bootstrapping paramétrico** (B = 2000) sobre las predicciones out-of-fold,
usando percentiles empíricos al 2.5 % y 97.5 %.

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────────────────────────
rows = []
for tag, m in [('A — Alerta Temprana (S1-S4)', m_A), ('B — Trayectoria Completa (S1-S8)', m_B)]:
    bs = m['bootstrap']
    rows.append({
        'Modelo': tag,
        'N': m['n_obs'],
        'Accuracy': f"{bs['accuracy_mean']:.4f}",
        'IC 95% Accuracy': f"[{bs['accuracy_ci_lo']:.4f}, {bs['accuracy_ci_hi']:.4f}]",
        'F1-macro': f"{bs['f1_macro_mean']:.4f}",
        'IC 95% F1': f"[{bs['f1_macro_ci_lo']:.4f}, {bs['f1_macro_ci_hi']:.4f}]",
    })

df_summary = pd.DataFrame(rows).set_index('Modelo')
print('=== Tabla Resumen de Desempeño ===')
display(df_summary)

In [ ]:
# ── Gráfica comparativa de IC ─────────────────────────────────────────────────
img = plt.imread(IMG_DIR / 'rf_comparativa_ic.png')
fig, ax = plt.subplots(figsize=(14, 6))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 2. Métricas por Clase con IC al 95 % (Bootstrap)

Se reportan Precisión y Recall para cada estado académico, con sus respectivos intervalos de confianza.

In [ ]:
def tabla_por_clase(m, modelo_tag):
    rows = []
    for clase, vals in m['bootstrap_per_class'].items():
        rows.append({
            'Clase': clase,
            'Precisión': f"{vals['precision_mean']:.3f}",
            'IC 95% Precisión': f"[{vals['precision_ci_lo']:.3f}, {vals['precision_ci_hi']:.3f}]",
            'Recall': f"{vals['recall_mean']:.3f}",
            'IC 95% Recall': f"[{vals['recall_ci_lo']:.3f}, {vals['recall_ci_hi']:.3f}]",
        })
    df = pd.DataFrame(rows).set_index('Clase')
    print(f'\n=== Métricas por Clase — Modelo {modelo_tag} ===')
    display(df)

tabla_por_clase(m_A, 'A (S1-S4)')
tabla_por_clase(m_B, 'B (S1-S8)')

In [ ]:
# ── Gráfica IC por clase ───────────────────────────────────────────────────────
COLORS = {
    'Titulado a tiempo':    '#2ecc71',
    'Titulado con rezago':  '#f39c12',
    'Abandono silencioso':  '#e74c3c',
    'Activo / Aún cursando':'#3498db',
}

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
fig.suptitle('Precisión y Recall por Clase — IC al 95 % (Bootstrap)', fontsize=14, y=1.01)

for row_idx, (m, tag) in enumerate([(m_A, 'Modelo A (S1-S4)'), (m_B, 'Modelo B (S1-S8)')]):
    for col_idx, metric in enumerate(['precision', 'recall']):
        ax = axes[row_idx][col_idx]
        cls_data = m['bootstrap_per_class']
        classes = list(cls_data.keys())
        means = [cls_data[c][f'{metric}_mean'] for c in classes]
        lo    = [cls_data[c][f'{metric}_ci_lo'] for c in classes]
        hi    = [cls_data[c][f'{metric}_ci_hi'] for c in classes]
        xerr  = [[m - l for m, l in zip(means, lo)],
                 [h - m for h, m in zip(hi, means)]]
        colors_list = [COLORS.get(c, '#95a5a6') for c in classes]
        bars = ax.barh(classes, means, color=colors_list, alpha=0.85, height=0.5)
        ax.errorbar(means, classes, xerr=xerr, fmt='none', color='black', capsize=5, linewidth=1.5)
        for bar, val in zip(bars, means):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=9)
        ax.set_xlim(0, 1.1)
        ax.set_title(f"{tag}\n{metric.capitalize()}", fontsize=11)
        ax.set_xlabel(metric.capitalize(), fontsize=10)
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## 3. Matrices de Confusión

Evaluadas sobre predicciones **out-of-fold** (5-fold CV estratificado), lo que evita el sobreajuste en la evaluación.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, m, fname, tag in [
    (axes[0], m_A, 'rf_confusion_matrix_s4.png', 'Modelo A — Alerta Temprana (S1-S4)'),
    (axes[1], m_B, 'rf_confusion_matrix_s8.png', 'Modelo B — Trayectoria Completa (S1-S8)'),
]:
    img = plt.imread(IMG_DIR / fname)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(tag, fontsize=12, pad=10)

plt.tight_layout()
plt.show()

---
## 4. Importancia de Variables (*Feature Importance*)

Las importancias Gini miden la reducción promedio de impureza que aporta cada variable al Random Forest.

**Código de colores:**
- 🔴 Interacción `esf × real` — variable de ingeniería clave
- 🔵 Índice de esfuerzo
- 🟢 Regularidad semestral
- 🟡 Promedio de calificaciones

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, fname, tag in [
    (axes[0], 'rf_feature_importance_s4.png', 'Modelo A — S1-S4'),
    (axes[1], 'rf_feature_importance_s8.png', 'Modelo B — S1-S8'),
]:
    img = plt.imread(IMG_DIR / fname)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(tag, fontsize=12, pad=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── Top 5 predictores de cada modelo en tabla ─────────────────────────────────
for tag, m in [('A (S1-S4)', m_A), ('B (S1-S8)', m_B)]:
    imp = pd.Series(m['importances']).sort_values(ascending=False).head(10)
    df_imp = pd.DataFrame({
        'Variable': imp.index,
        'Importancia Gini': imp.values.round(4),
        'Rango (%)': (imp.values / imp.values.sum() * 100).round(1),
    })
    print(f'\n=== Top 10 Predictores — Modelo {tag} ===')
    display(df_imp.reset_index(drop=True))

---
## 5. Prueba de Hipótesis: Test de McNemar (Modelo A vs Modelo B)

**Hipótesis:**
- **H₀:** El Modelo A y el Modelo B tienen la misma tasa de error (los errores son simétricos).
- **H₁:** Los modelos difieren significativamente en su capacidad predictiva.

La prueba de McNemar trabaja sobre pares de observaciones comunes a ambos modelos,
analizando los casos donde uno acierta y el otro falla.

Se aplica la **corrección de continuidad de Yates** (McNemar corregido).

In [ ]:
print('='*55)
print('       PRUEBA DE McNEMAR — Modelo A vs Modelo B')
print('='*55)

b = mc['b']  # A acierta, B falla
c = mc['c']  # A falla, B acierta

print(f"\nTabla de contingencia de discordancias:")
print(f"  b (A bien, B mal) = {b}")
print(f"  c (A mal, B bien) = {c}")
print(f"\nEstadístico χ² (con corrección Yates): {mc['statistic']:.4f}")
print(f"P-valor:                                {mc['p_value']:.4e}")
print(f"\n{'─'*55}")
print(f"Decisión: {mc['decision']}")
print(f"{'─'*55}")

alpha = 0.05
if mc['p_value'] < alpha:
    print(f"\n✅ Con α = {alpha}: El Modelo B (S1-S8) supera significativamente")
    print(f"   al Modelo A (S1-S4). Agregar los semestres S5-S8 mejora")
    print(f"   la precisión de manera estadísticamente robusta.")
    print(f"\n   Interpretación de b vs c:")
    if b > c:
        print(f"   El Modelo B recupera {c} casos que A no podía predecir,")
        print(f"   pero A mantiene ventaja en {b} casos (probablemente 'Activos').")
    else:
        print(f"   El Modelo B corrige {c} errores que A cometía.")

In [ ]:
# ── Visualización de la tabla de McNemar ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

tabla = np.array([
    ['Ambos correctos', f'A bien, B mal\nb={mc["b"]}'],
    [f'A mal, B bien\nc={mc["c"]}', 'Ambos incorrectos'],
])

colors = [
    ['#d5f5e3', '#fdebd0'],
    ['#d6eaf8', '#fadbd8'],
]

ax.axis('off')
tbl = ax.table(
    cellText=tabla,
    rowLabels=['Modelo A correcto', 'Modelo A incorrecto'],
    colLabels=['Modelo B correcto', 'Modelo B incorrecto'],
    cellLoc='center', loc='center',
    cellColours=colors,
)
tbl.scale(1.6, 2.2)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)

ax.set_title(
    f'Tabla de McNemar: Modelo A vs Modelo B\n'
    f'χ² = {mc["statistic"]:.2f}, p = {mc["p_value"]:.2e}',
    fontsize=13, pad=20,
)
plt.tight_layout()
plt.show()

---
## 6. Síntesis de Hallazgos

| Dimensión | Modelo A (S1-S4) | Modelo B (S1-S8) |
|-----------|-----------------|------------------|
| **Observaciones** | 4,853 | 4,276 |
| **Accuracy** | — | — |
| **F1-macro** | — | — |
| **Caso mejor predicho** | Titulado a tiempo | Titulado a tiempo |
| **Caso peor predicho** | Activo/Cursando | Activo/Cursando |

> **Conclusión estadística:** La prueba de McNemar rechaza la hipótesis de equivalencia entre modelos. 
> El Modelo B, con la trayectoria completa hasta S8, ofrece una capacidad predictiva 
> significativamente superior, lo que justifica el diseño de un **sistema de alerta tardía** 
> además del sistema de alerta temprana basado en S1-S4.

In [ ]:
# ── Tabla final automática con valores reales ─────────────────────────────────
rows_final = []
for tag, m in [('A (S1-S4)', m_A), ('B (S1-S8)', m_B)]:
    bs = m['bootstrap']
    top_cls = max(m['bootstrap_per_class'].items(),
                  key=lambda x: x[1]['recall_mean'])[0]
    bot_cls = min(m['bootstrap_per_class'].items(),
                  key=lambda x: x[1]['recall_mean'])[0]
    rows_final.append({
        'Modelo': f'Modelo {tag}',
        'N': m['n_obs'],
        'Accuracy (IC 95%)': f"{bs['accuracy_mean']:.3f} [{bs['accuracy_ci_lo']:.3f}–{bs['accuracy_ci_hi']:.3f}]",
        'F1-macro (IC 95%)': f"{bs['f1_macro_mean']:.3f} [{bs['f1_macro_ci_lo']:.3f}–{bs['f1_macro_ci_hi']:.3f}]",
        'Mejor clase (Recall)': top_cls,
        'Peor clase (Recall)': bot_cls,
    })

df_final = pd.DataFrame(rows_final).set_index('Modelo')
print('=== Tabla Final de Resultados ===')
display(df_final)